# 03 Technical cohorts

Both `GET /api/{market}/technical-strategies/cohorts` and `GET /api/us/layoffs/latest` are member research routes (beta).

This notebook fetches the US cohort feed once by default. Set `SURGEFLOW_COHORT_MARKET` to `cn`, `jp`, or `hk` to select another market.

The source date and data status appear above the table. HTTP 503 with `COHORT_FEED_UNAVAILABLE` shows **Feed unavailable**, not an error.

Entry prices are historical references, not executed prices. Observation fields are historical measurements. Research data only, not investment advice.

The coverage counts describe companies watched, companies without the required turnover change, and companies with missing inputs.

Free access grants no redistribution rights. **Do not republish the data you fetch.**

In [ ]:
import os
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetch_payload, get_api_key, make_client, now_utc, show_panel
from surgeflow.monitor import past_valid_until
client, catalogue = make_client()
api_key = get_api_key()
MARKET = os.environ.get('SURGEFLOW_COHORT_MARKET', 'us').strip().lower()
cohorts_payload = fetch_payload(
    f'03 Technical cohorts — {MARKET}',
    lambda: client.member_technical_cohorts(MARKET, api_key),
)

In [ ]:
COHORT_COLUMNS = {
    'cohort_id': 'The unique identifier for this dated research cohort.',
    'strategy_id': 'The published rule set that formed this cohort.',
    'market': 'The equity market for this cohort.',
    'currency': 'The currency of the reference prices and cash amounts.',
    'weighting': 'EW means equal weight; ERC means equal risk contribution.',
    'cohort_state': 'The publication state of this cohort and its inputs.',
    'as_of': 'The cohort formation date.',
    'built_at': 'The time when the producer built this cohort.',
    'ticker': 'The company identifier in the selected market.',
    'entry_date': 'The date attached to the entry reference price.',
    'entry_reference_price': 'The verified reference close, not an executed price.',
    'starting_weight': 'The cohort weight after share rounding and cash retention.',
    'target_weight': 'The rule-based weight before share rounding.',
    'reference_shares': 'The whole-share quantity used for reference calculations.',
    'observation_as_of': 'The date of the latest published observation.',
    'observation_state': 'The availability state of the observation.',
    'elapsed_sessions': 'The number of observed sessions since cohort entry.',
    'observation_window': 'The state of the fixed observation window.',
    'gross_price_return': 'The historical cohort price change before costs and dividends.',
    'gross_usd_return': 'The historical cohort price change in US dollars, including FX where supplied.',
    'holding_price_return': 'The historical price change for this holding since entry.',
    'holding_session_return': 'The historical price change for this holding in the latest session.',
    'holding_observation_state': 'The availability state for this holding observation.',
}
def cohort_rows(payload):
    rows = []
    for cohort in payload.get('cohorts') or []:
        observation = cohort.get('observation') or {}
        observed = {row.get('ticker'): row for row in observation.get('holdings') or []}
        for holding in cohort.get('holdings') or [None]:
            holding = holding or {}
            observed_row = observed.get(holding.get('ticker'), {})
            rows.append({
                'cohort_id': cohort.get('cohort_id'),
                'strategy_id': cohort.get('strategy_id'),
                'market': cohort.get('market') or payload.get('market', 'us'),
                'currency': cohort.get('currency') or ('USD' if (cohort.get('market') or payload.get('market') or 'us') == 'us' else None),
                'weighting': cohort.get('weighting'),
                'cohort_state': cohort.get('state'),
                'as_of': cohort.get('as_of'),
                'built_at': cohort.get('built_at'),
                'ticker': holding.get('ticker'),
                'entry_date': holding.get('entry_date'),
                'entry_reference_price': holding.get('entry_reference_price'),
                'starting_weight': holding.get('starting_weight'),
                'target_weight': holding.get('target_weight'),
                'reference_shares': holding.get('reference_shares'),
                'observation_as_of': observation.get('as_of'),
                'observation_state': observation.get('state'),
                'elapsed_sessions': observation.get('elapsed_sessions'),
                'observation_window': observation.get('observation_window'),
                'gross_price_return': observation.get('gross_price_return'),
                'gross_usd_return': observation.get('gross_usd_return'),
                'holding_price_return': observed_row.get('price_return'),
                'holding_session_return': observed_row.get('session_return'),
                'holding_observation_state': observed_row.get('state'),
            })
    return rows
if cohorts_payload is not None:
    coverage = cohorts_payload.get('coverage') or {}
    message = (
        f"Published at: {cohorts_payload.get('published_at', 'not supplied')}; "
        f"valid until: {cohorts_payload.get('valid_until', 'not supplied')}; "
        f"companies watched: {coverage.get('roster', 'not supplied')}; "
        f"turnover change not met: {coverage.get('turnover_not_met', 'not supplied')}; "
        f"missing inputs: {coverage.get('missing_inputs', 'not supplied')}"
    )
    cohort_frame = show_panel(
        f'03 Technical cohorts — {MARKET}',
        cohorts_payload,
        cohort_rows(cohorts_payload),
        COHORT_COLUMNS,
        source_date=cohorts_payload.get('as_of') or cohorts_payload.get('published_at'),
        status=(
            'not supplied' if cohorts_payload.get('ok') is not True
            else 'past valid_until' if past_valid_until(cohorts_payload.get('valid_until'), now_utc())
            else 'available'
        ),
        message=message,
    )
    print('Feed limitations:')
    for limitation in cohorts_payload.get('limitations') or ['not supplied']:
        print(f'- {limitation}')